In [1]:
from importlib import reload
from pathlib import Path
import sys

import torch
import transformers as tr
import transformer_lens as tl

for src_path in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if src_path.exists() and str(src_path) not in sys.path:
        sys.path.append(str(src_path))

import core.duplicate.utils as duplicate_utils
import core.duplicate.visualization as duplicate_visualization
reload(duplicate_utils)
reload(duplicate_visualization)

from data import get_duplicate_token_data
from core.duplicate import (
    calculate_duplicate_attention_scores,
    duplicate_logit_diff_metric,
    find_top_duplicate_heads,
    patch_duplicate_heads,
    validate_duplicate_token_alignment,
)
from core.duplicate.visualization import show_duplicate_attention


In [2]:
gpt2_tokenizer = tr.GPT2Tokenizer.from_pretrained("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

gpt2 = tl.HookedTransformer.from_pretrained("gpt2", device=device, dtype=dtype)
gpt2.eval()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

In [3]:
dup_data = get_duplicate_token_data(tokenizer=gpt2_tokenizer, device=device)
clean_tokens = dup_data["clean_tokens"]
corrupted_tokens = dup_data["corrupted_tokens"]
metadata = dup_data["metadata"]

alignment = validate_duplicate_token_alignment(metadata)
alignment


{'first_repeat_token_index': [2],
 'second_repeat_token_index': [10],
 'prediction_token_index': [14],
 'is_aligned': True}

In [4]:
sample_idx = 0
sample_tokens = [
    gpt2_tokenizer.decode([token_id])
    for token_id in clean_tokens["input_ids"][sample_idx].detach().cpu().tolist()
]

list(enumerate(sample_tokens)), metadata[sample_idx]


([(0, 'Yesterday'),
  (1, ','),
  (2, ' John'),
  (3, ' and'),
  (4, ' Mary'),
  (5, ' went'),
  (6, ' to'),
  (7, ' the'),
  (8, ' store'),
  (9, '.'),
  (10, ' John'),
  (11, ' gave'),
  (12, ' a'),
  (13, ' drink'),
  (14, ' to')],
 {'clean': 'Yesterday, John and Mary went to the store. John gave a drink to',
  'corrupted': 'Yesterday, John and Mary went to the store. Paul gave a drink to',
  'first_name': 'John',
  'second_name': 'Mary',
  'clean_repeated_name': 'John',
  'corrupted_name': 'Paul',
  'structure': 'ABA',
  'correct_name': 'Mary',
  'incorrect_name': 'John',
  'prediction_token_index': 14,
  'first_repeat_char_start': 11,
  'first_repeat_char_end': 15,
  'second_repeat_char_start': 44,
  'second_repeat_char_end': 48,
  'corrupted_subject_char_start': 44,
  'corrupted_subject_char_end': 48,
  'first_repeat_token_span': [2],
  'second_repeat_token_span': [10],
  'corrupted_subject_token_span': [10],
  'first_repeat_token_index': 2,
  'second_repeat_token_index': 10,
  '

In [5]:
names_filter = lambda name: (
    name.endswith("attn.hook_pattern")
    or name.endswith("attn.hook_z")
)

with torch.inference_mode():
    clean_logits, clean_cache = gpt2.run_with_cache(
        clean_tokens["input_ids"],
        names_filter=names_filter,
    )
    corrupted_logits, corrupted_cache = gpt2.run_with_cache(
        corrupted_tokens["input_ids"],
        names_filter=names_filter,
    )


In [6]:
duplicate_scores = calculate_duplicate_attention_scores(clean_cache, metadata)
top_heads = find_top_duplicate_heads(clean_cache, metadata, top_k=20)
top_heads


,layer,head,block,duplicate_attention_score
0,1,11,blocks.1.attn.hook_pattern,0.686738
1,3,0,blocks.3.attn.hook_pattern,0.677830
2,0,5,blocks.0.attn.hook_pattern,0.600340
3,0,1,blocks.0.attn.hook_pattern,0.532705
4,0,10,blocks.0.attn.hook_pattern,0.271418
5,0,6,blocks.0.attn.hook_pattern,0.174246
6,0,2,blocks.0.attn.hook_pattern,0.122907
7,0,0,blocks.0.attn.hook_pattern,0.110912
8,1,6,blocks.1.attn.hook_pattern,0.103397
9,4,7,blocks.4.attn.hook_pattern,0.093827


In [7]:
top_layers = (
    top_heads.groupby("layer")["duplicate_attention_score"]
    .max()
    .sort_values(ascending=False)
    .head(3)
)

top_layers


layer
1    0.686738
3    0.677830
0    0.600340
Name: duplicate_attention_score, dtype: float64

In [8]:
top_head = top_heads.iloc[0]
show_duplicate_attention(
    cache=clean_cache,
    tokens=clean_tokens,
    tokenizer=gpt2_tokenizer,
    layer=int(top_head["layer"]),
    batch_idx=sample_idx,
    heads=[int(top_head["head"])],
)


In [9]:
best_layer = int(top_layers.index[0])
best_layer_heads = (
    top_heads[top_heads["layer"] == best_layer]["head"]
    .astype(int)
    .head(4)
    .tolist()
)

show_duplicate_attention(
    cache=clean_cache,
    tokens=clean_tokens,
    tokenizer=gpt2_tokenizer,
    layer=best_layer,
    batch_idx=sample_idx,
    heads=best_layer_heads,
)


In [10]:
clean_metric = duplicate_logit_diff_metric(clean_logits, clean_tokens, metadata)
corrupted_metric = duplicate_logit_diff_metric(corrupted_logits, corrupted_tokens, metadata)

{
    "clean_metric": float(clean_metric.detach().cpu()),
    "corrupted_metric": float(corrupted_metric.detach().cpu()),
}


{'clean_metric': 4.705811977386475, 'corrupted_metric': -0.5886232256889343}

In [11]:
top_heads_to_patch = top_heads[["layer", "head"]].head(5).to_dict("records")

head_patch_result = patch_duplicate_heads(
    model=gpt2,
    clean_tokens=clean_tokens,
    corrupted_tokens=corrupted_tokens,
    metadata=metadata,
    heads=top_heads_to_patch,
    patch_positions="second_repeat",
)

head_patch_result


{'clean_metric': 4.705811977386475,
 'corrupted_metric': -0.5886232256889343,
 'patched_metric': -0.06004648655653,
 'recovery': 0.09983628988265991,
 'heads': [{'layer': 1, 'head': 11},
  {'layer': 3, 'head': 0},
  {'layer': 0, 'head': 5},
  {'layer': 0, 'head': 1},
  {'layer': 0, 'head': 10}],
 'patch_positions': 'second_repeat'}

In [12]:
layers_to_patch = [int(layer) for layer in top_layers.index[:2]]

layer_patch_result = patch_duplicate_heads(
    model=gpt2,
    clean_tokens=clean_tokens,
    corrupted_tokens=corrupted_tokens,
    metadata=metadata,
    layers=layers_to_patch,
    patch_positions="second_repeat",
)

layer_patch_result


{'clean_metric': 4.705811977386475,
 'corrupted_metric': -0.5886232256889343,
 'patched_metric': -0.07548396289348602,
 'recovery': 0.09692049026489258,
 'heads': [{'layer': 1, 'head': 0},
  {'layer': 1, 'head': 1},
  {'layer': 1, 'head': 2},
  {'layer': 1, 'head': 3},
  {'layer': 1, 'head': 4},
  {'layer': 1, 'head': 5},
  {'layer': 1, 'head': 6},
  {'layer': 1, 'head': 7},
  {'layer': 1, 'head': 8},
  {'layer': 1, 'head': 9},
  {'layer': 1, 'head': 10},
  {'layer': 1, 'head': 11},
  {'layer': 3, 'head': 0},
  {'layer': 3, 'head': 1},
  {'layer': 3, 'head': 2},
  {'layer': 3, 'head': 3},
  {'layer': 3, 'head': 4},
  {'layer': 3, 'head': 5},
  {'layer': 3, 'head': 6},
  {'layer': 3, 'head': 7},
  {'layer': 3, 'head': 8},
  {'layer': 3, 'head': 9},
  {'layer': 3, 'head': 10},
  {'layer': 3, 'head': 11}],
 'patch_positions': 'second_repeat'}

In [13]:
patch_summary = {
    "top_heads": top_heads_to_patch,
    "top_layers": layers_to_patch,
    "top_head_recovery": head_patch_result["recovery"],
    "top_layer_recovery": layer_patch_result["recovery"],
}

patch_summary


{'top_heads': [{'layer': 1, 'head': 11},
  {'layer': 3, 'head': 0},
  {'layer': 0, 'head': 5},
  {'layer': 0, 'head': 1},
  {'layer': 0, 'head': 10}],
 'top_layers': [1, 3],
 'top_head_recovery': 0.09983628988265991,
 'top_layer_recovery': 0.09692049026489258}